In [1]:
!python --version  

Python 3.13.12


In [2]:
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Load env variables
from dotenv import load_dotenv
load_dotenv()

# avoid warnings for old claude model use
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [4]:
# Initialize Antropic Chat client
from anthropic import Anthropic
client = Anthropic()

In [5]:
# Helper Chat Functions


## Function to Append User messasge
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


## Function to Append Assistant messasge
def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


## Main Function for chat
def get_response(messages, stop_sequences=None, system=None):

    model = "claude-haiku-4-5"  # you can change the model according to your usecase
    params = {
        "model": model,
        "max_tokens": 6000,
        "messages": messages,
    }
    if system:
        params["system"] = system    
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    message = client.messages.create(**params)
    return message


def get_chat_content(message):
    return message.content[0].text

In [6]:
def getStructureOutput(mesg, data_prefix="```json", stop_sequences=None):
    messages = []

    # User's request > what user wants in specific format
    add_user_message(messages, mesg)

    # start the content and make the AI generate the remaining part.
    add_assistant_message(messages, data_prefix)

    response = get_response(messages=messages, stop_sequences=stop_sequences)
    return get_chat_content(response)

In [7]:

import json  # For json format validation
import re    # For regex format validation
import ast   # For python syntax validation

def validate_json(data):
    try:
        json.load(data.strip())
        return 10
    except json.JSONDecodeError: return 0

def validate_python(code):
    try:
        ast.parse(code.strip())
        return 10
    except: return 0 

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except: return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [8]:
import json
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    
    eval_text = getStructureOutput(eval_prompt ,stop_sequences=['```'])
    print(eval_text)
    return json.loads(eval_text)

In [9]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""

    output = getStructureOutput(
        prompt,
        data_prefix="```code",  # a little cheat code to cover all the formats
        stop_sequences=["```"],
    )
    return output


def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    strengths = model_grade["strengths"]
    weaknesses = model_grade["weaknesses"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "weaknesses": weaknesses,
        "strengths": strengths,
        "reasoning": reasoning,
    }


from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [10]:
import json

with open("./json_db/dataset.json","r") as db:
    results = run_eval(json.load(db))


{
    "strengths": [
        "Proper input validation with clear error messages and type checking for instance_id parameter",
        "Comprehensive error handling with specific ClientError catching and detailed error information extraction",
        "Well-documented function with docstring explaining parameters, return value, and exceptions"
    ],
    "weaknesses": [
        "Excessive print statements mixed with exception handling reduce code reusability in production/library contexts; logging should be used instead",
        "No validation of instance_id format (should match pattern like 'i-*'); empty string check is insufficient",
        "Missing boto3 client region specification, which could lead to unexpected behavior if default region isn't configured"
    ],
    "reasoning": "The solution correctly implements the core requirement of starting an EC2 instance with boto3 and includes good defensive programming practices. However, it has practical production concerns: print stat

In [11]:
# With less temprature we will likely get the almost same responses each time

print(json.dumps(results,indent=2))


[
  {
    "output": "\nimport boto3\nfrom botocore.exceptions import ClientError\n\ndef start_ec2_instance(instance_id: str) -> dict:\n    \"\"\"\n    Start an Amazon EC2 instance given an instance ID.\n    \n    Args:\n        instance_id (str): The ID of the EC2 instance to start\n        \n    Returns:\n        dict: Response from the start_instances API call\n        \n    Raises:\n        ClientError: If there's an error starting the instance\n        ValueError: If instance_id is empty or invalid\n    \"\"\"\n    \n    # Validate input\n    if not instance_id or not isinstance(instance_id, str):\n        raise ValueError(\"instance_id must be a non-empty string\")\n    \n    # Create EC2 client\n    ec2_client = boto3.client('ec2')\n    \n    try:\n        # Start the instance\n        response = ec2_client.start_instances(InstanceIds=[instance_id])\n        \n        print(f\"Successfully started instance: {instance_id}\")\n        print(f\"Response: {response}\")\n        \n   